# LAB 3
- Sử dụng bộ dữ liệu IndoorOutdoorNet-20k (kaggle) để làm cơ sở dữ liệu huấn luyện cho mô hình FCN có 5 hidden layer, có kết hợp cơ chế back drop:
  1. Sử dụng entropyloss cho hàm loss
  2. Sử dụng cơ chế huấn luyện mini batch
  3. Kết hợp cơ chế sốc cho hàm loss: cứ 5 epoch giảm 10% loss. (lưu ý số epcoh tối đa không vượt quá 60)
  4. Đưa ra dự báo 1 vài kết quả.

In [ ]:
import os
import random
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score
from google.colab import userdata

# Cấu hình thiết bị (GPU/CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Đang sử dụng thiết bị: {device}")

In [ ]:
def download_and_prepare_data():
    """Tải dataset từ Kaggle bằng Colab Secrets"""
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

    if not os.path.exists('dataset'):
        !kaggle datasets download -d prithivsakthiur/indooroutdoornet-20k
        !unzip -q indooroutdoornet-20k.zip -d dataset/
        print("Đã tải và giải nén dữ liệu thành công!")
    else:
        print("Dữ liệu đã tồn tại, bỏ qua bước tải.")

# Thực thi tải data
download_and_prepare_data()

In [ ]:
# Cấu hình hằng số
DATA_DIR = 'dataset/Indoor vs Outdoor'
BATCH_SIZE = 256
IMG_SIZE = (64, 64)
INPUT_SIZE = IMG_SIZE[0] * IMG_SIZE[1] * 3
VAL_SPLIT = 0.1
TEST_SPLIT = 0.2

def create_loaders(data_dir, batch_size=32, img_size=(64, 64), val_split=0.1, test_split=0.2):
    """Tạo DataLoader cho quá trình train, validation và test (Cơ chế Mini-batch)"""
    transform = transforms.Compose([
        transforms.Resize(img_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
    ])

    dataset = datasets.ImageFolder(root=data_dir, transform=transform)

    # Tính toán kích thước các tập dữ liệu
    total_size = len(dataset)
    test_size = int(total_size * test_split)
    val_size = int(total_size * val_split)
    train_size = total_size - test_size - val_size

    # Chia tập dữ liệu
    train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, val_size, test_size])

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, test_loader, dataset.classes

# Khởi tạo DataLoader
train_loader, val_loader, test_loader, classes = create_loaders(DATA_DIR, BATCH_SIZE, IMG_SIZE, VAL_SPLIT, TEST_SPLIT)
NUM_CLASSES = len(classes)
print(f"Các nhãn phân loại: {classes}")

In [ ]:
class FCN(nn.Module):
    """Mô hình Fully Connected Network với 5 Hidden Layers và cơ chế Back drop (Dropout)"""
    def __init__(self, input_size, num_classes):
        super(FCN, self).__init__()
        self.flatten = nn.Flatten()

        self.network = nn.Sequential(
            nn.Linear(input_size, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = self.flatten(x)
        return self.network(x)

In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer, epochs=60):
    train_loss_history = []
    val_loss_history = []

    # Giới hạn số epoch tối đa không vượt quá 60
    epochs = min(epochs, 60)

    for epoch in range(epochs):
        # --- BƯỚC 1: TRAINING ---
        model.train()
        running_train_loss = 0.0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            # Cơ chế "sốc loss": Cứ 5 epoch thì giảm 10% giá trị loss trước khi backward
            if (epoch + 1) % 5 == 0:
                loss = loss * 0.9

            loss.backward()
            optimizer.step()
            running_train_loss += loss.item()

        avg_train_loss = running_train_loss / len(train_loader)
        train_loss_history.append(avg_train_loss)

        # --- BƯỚC 2: VALIDATION ---
        model.eval() # Tắt dropout khi đánh giá
        running_val_loss = 0.0
        correct = 0
        total = 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)

                loss = criterion(outputs, labels)
                running_val_loss += loss.item()

                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        avg_val_loss = running_val_loss / len(val_loader)
        val_loss_history.append(avg_val_loss)
        val_acc = 100 * correct / total

        print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.2f}%")

    return train_loss_history, val_loss_history

In [ ]:
def plot_loss(train_loss, val_loss):
    """Trực quan hóa Loss của Train và Validation"""
    plt.figure(figsize=(10, 5))
    plt.plot(train_loss, marker='o', color='b', label='Training Loss')
    plt.plot(val_loss, marker='x', color='r', label='Validation Loss')
    plt.title("Biểu đồ suy giảm Loss qua các Epochs")
    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.grid(True)
    plt.legend()
    plt.show()

def evaluate_model_comprehensive(model, test_loader, classes):
    """Đánh giá toàn diện các chỉ số Accuracy, Precision, Recall, F1-Score"""
    model.eval()
    all_preds = []
    all_labels = []

    print("\nĐang tiến hành đánh giá toàn diện mô hình trên tập Test...")
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average='macro', zero_division=0)
    recall = recall_score(all_labels, all_preds, average='macro', zero_division=0)
    f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)

    print("="*45)
    print(" BÁO CÁO KẾT QUẢ ĐÁNH GIÁ TỔNG THỂ")
    print("="*45)
    print(f" Accuracy  : {acc * 100:.2f}%")
    print(f" Precision : {precision:.4f}")
    print(f" Recall    : {recall:.4f}")
    print(f" F1-Score  : {f1:.4f}")
    print("-" * 45)
    print("Báo cáo chi tiết từng phân lớp:")
    print(classification_report(all_labels, all_preds, target_names=classes, zero_division=0))

def predict(model, image_tensor):
    """Hàm dự đoán cho 1 ảnh (Tensor)"""
    model.eval()
    with torch.no_grad():
        image_tensor = image_tensor.unsqueeze(0).to(device)
        outputs = model(image_tensor)
        _, predicted = torch.max(outputs.data, 1)
        return predicted.item()

def demo_test_random(model, test_loader, classes, n=5):
    """Chọn ngẫu nhiên ảnh để test và hiển thị"""
    dataset = test_loader.dataset
    indices = random.sample(range(len(dataset)), n)

    # Đưa mean/std vào cục bộ để tránh lỗi scope
    mean = np.array([0.5, 0.5, 0.5])
    std = np.array([0.5, 0.5, 0.5])

    plt.figure(figsize=(15, 3))
    for i, idx in enumerate(indices):
        image, label = dataset[idx]
        pred_idx = predict(model, image)

        # Denormalize
        img_display = image.numpy().transpose((1, 2, 0))
        img_display = std * img_display + mean
        img_display = np.clip(img_display, 0, 1)

        plt.subplot(1, n, i + 1)
        plt.imshow(img_display)
        gt_name = classes[label]
        pred_name = classes[pred_idx]

        color = 'green' if gt_name == pred_name else 'red'
        plt.title(f"GT: {gt_name}\nPred: {pred_name}", color=color)
        plt.axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
print(f"Số lượng mẫu tập Train: {len(train_loader.dataset)}")
print(f"Số lượng mẫu tập Validation: {len(val_loader.dataset)}")
print(f"Số lượng mẫu tập Test: {len(test_loader.dataset)}")

In [ ]:
# Khởi tạo Model, Loss (EntropyLoss) và Optimizer
model = FCN(INPUT_SIZE, NUM_CLASSES).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)
EPOCHS = 30


print("BẮT ĐẦU HUẤN LUYỆN...")
train_history, val_history = train_model(model, train_loader, val_loader, criterion, optimizer, epochs=EPOCHS)

In [ ]:
# 3. Vẽ biểu đồ Loss
plot_loss(train_history, val_history)

In [ ]:
# 4. In báo cáo các chỉ số F1, Recall, Precision
evaluate_model_comprehensive(model, test_loader, classes)

In [ ]:
# 5. Demo dự đoán bằng hình ảnh trực quan
print("\nDEMO DỰ ĐOÁN NGẪU NHIÊN:")
demo_test_random(model, test_loader, classes, n=10)

In [ ]:
# Neu widget chua hien thi duoc tren Colab thi mo manager
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

import math
import random
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import torch
import ipywidgets as widgets
from IPython.display import display, clear_output


def build_test_cache(model, test_loader, device):
    """
    Chay infer 1 lan tren toan bo test set va cache ket qua:
    idx trong test subset, nhan that, nhan du doan, confidence.
    """
    model.eval()
    cache = []
    idx_offset = 0

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            logits = model(inputs)
            probs = torch.softmax(logits, dim=1)
            confs, preds = torch.max(probs, dim=1)

            labels_np = labels.cpu().numpy()
            preds_np = preds.cpu().numpy()
            confs_np = confs.cpu().numpy()

            for i in range(len(labels_np)):
                cache.append(
                    {
                        "idx": idx_offset + i,       # index trong test_loader.dataset
                        "gt": int(labels_np[i]),     # ground truth
                        "pred": int(preds_np[i]),    # prediction
                        "conf": float(confs_np[i]),  # confidence [0,1]
                    }
                )

            idx_offset += len(labels_np)

    return cache


def demo_test_pro(model, test_loader, classes, default_page_size=10):
    dataset = test_loader.dataset
    if len(dataset) == 0:
        print("Test set rong, khong the demo.")
        return

    print("Dang tao prediction cache... (chi chay 1 lan)")
    cache = build_test_cache(model, test_loader, device)
    print(f"Da cache {len(cache)} mau test.")

    mean = np.array([0.5, 0.5, 0.5])
    std = np.array([0.5, 0.5, 0.5])
    class_to_idx = {name: i for i, name in enumerate(classes)}

    # ----- Widgets -----
    btn_prev = widgets.Button(description="Prev", icon="arrow-left")
    btn_next = widgets.Button(description="Next", icon="arrow-right")
    btn_refresh = widgets.Button(description="Refresh", icon="refresh", button_style="info")

    page_size = widgets.IntSlider(
        value=min(default_page_size, 24),
        min=4,
        max=min(24, len(cache)),
        step=1,
        description="Anh/trang",
        continuous_update=False,
    )

    class_filter = widgets.Dropdown(
        options=["All"] + classes,
        value="All",
        description="Class",
    )

    only_wrong = widgets.Checkbox(
        value=False,
        description="Chi anh doan sai",
    )

    min_conf = widgets.FloatSlider(
        value=0.0,
        min=0.0,
        max=1.0,
        step=0.01,
        readout_format=".2f",
        description="Min conf",
        continuous_update=False,
    )

    sort_mode = widgets.Dropdown(
        options=[
            ("Ngau nhien", "random"),
            ("Conf thap -> cao", "asc"),
            ("Conf cao -> thap", "desc"),
        ],
        value="random",
        description="Sap xep",
    )

    seed_box = widgets.IntText(
        value=-1,  # -1 = random moi lan refresh
        description="Seed",
    )

    page_info = widgets.HTML()
    stats_info = widgets.HTML()
    out = widgets.Output()

    state = {
        "pool": [],
        "page": 0,
        "nonce": 0,  # tang moi lan refresh de random ra bo moi
    }

    def apply_filters():
        pool = cache

        if class_filter.value != "All":
            cls_idx = class_to_idx[class_filter.value]
            pool = [r for r in pool if r["gt"] == cls_idx]

        if only_wrong.value:
            pool = [r for r in pool if r["pred"] != r["gt"]]

        pool = [r for r in pool if r["conf"] >= min_conf.value]

        if sort_mode.value == "asc":
            pool = sorted(pool, key=lambda x: x["conf"])
        elif sort_mode.value == "desc":
            pool = sorted(pool, key=lambda x: x["conf"], reverse=True)
        else:
            pool = pool[:]
            if seed_box.value < 0:
                random.shuffle(pool)
            else:
                rng = random.Random(seed_box.value + state["nonce"])
                rng.shuffle(pool)

        return pool

    def update_stats(pool):
        n = len(pool)
        if n == 0:
            stats_info.value = "<b>Mau sau loc:</b> 0"
            return

        wrong = sum(1 for r in pool if r["gt"] != r["pred"])
        wrong_rate = 100.0 * wrong / n
        avg_conf = 100.0 * sum(r["conf"] for r in pool) / n

        stats_info.value = (
            f"<b>Mau sau loc:</b> {n} | "
            f"<b>Sai:</b> {wrong} ({wrong_rate:.1f}%) | "
            f"<b>Avg conf:</b> {avg_conf:.1f}%"
        )

    def render():
        pool = state["pool"]
        n = len(pool)
        per_page = page_size.value
        total_pages = max(1, math.ceil(n / per_page))

        state["page"] = min(state["page"], total_pages - 1)
        state["page"] = max(0, state["page"])

        btn_prev.disabled = state["page"] <= 0
        btn_next.disabled = state["page"] >= total_pages - 1

        page_info.value = f"<b>Trang:</b> {state['page'] + 1}/{total_pages}"
        update_stats(pool)

        start = state["page"] * per_page
        end = start + per_page
        current = pool[start:end]

        with out:
            clear_output(wait=True)

            if len(current) == 0:
                print("Khong co anh nao phu hop bo loc hien tai.")
                return

            cols = min(4, len(current))
            rows = math.ceil(len(current) / cols)

            plt.figure(figsize=(4.2 * cols, 3.8 * rows))
            wrong_pairs = []

            for i, item in enumerate(current, start=1):
                image, _ = dataset[item["idx"]]

                img = image.detach().cpu().numpy().transpose((1, 2, 0))
                img = std * img + mean
                img = np.clip(img, 0, 1)

                gt_name = classes[item["gt"]]
                pred_name = classes[item["pred"]]
                is_correct = item["gt"] == item["pred"]

                if not is_correct:
                    wrong_pairs.append((gt_name, pred_name))

                color = "green" if is_correct else "crimson"

                plt.subplot(rows, cols, i)
                plt.imshow(img)
                plt.title(
                    f"GT: {gt_name}\nPred: {pred_name} ({item['conf']*100:.1f}%)",
                    color=color,
                    fontsize=10,
                )
                plt.axis("off")

            plt.tight_layout()
            plt.show()

            if wrong_pairs:
                top_confusions = Counter(wrong_pairs).most_common(5)
                print("Top nham lan tren trang hien tai:")
                for (gt, pred), c in top_confusions:
                    print(f"- {gt} -> {pred}: {c}")
            else:
                print("Trang nay khong co mau doan sai.")

    def rebuild_pool(reset_page=True, bump_nonce=False):
        if bump_nonce:
            state["nonce"] += 1
        state["pool"] = apply_filters()
        if reset_page:
            state["page"] = 0
        render()

    def on_prev(_):
        state["page"] = max(0, state["page"] - 1)
        render()

    def on_next(_):
        per_page = page_size.value
        last = max(0, math.ceil(len(state["pool"]) / per_page) - 1)
        state["page"] = min(last, state["page"] + 1)
        render()

    def on_control_change(_):
        rebuild_pool(reset_page=True, bump_nonce=False)

    btn_prev.on_click(on_prev)
    btn_next.on_click(on_next)
    btn_refresh.on_click(lambda _: rebuild_pool(reset_page=True, bump_nonce=True))

    for w in [page_size, class_filter, only_wrong, min_conf, sort_mode, seed_box]:
        w.observe(on_control_change, names="value")

    controls_row_1 = widgets.HBox([btn_prev, btn_next, btn_refresh, page_info])
    controls_row_2 = widgets.HBox([page_size, class_filter, only_wrong])
    controls_row_3 = widgets.HBox([sort_mode, min_conf, seed_box])

    display(widgets.VBox([controls_row_1, controls_row_2, controls_row_3, stats_info, out]))

    # Lan dau tien: random 1 bo mau
    rebuild_pool(reset_page=True, bump_nonce=True)


# Chay demo
print("\nDEMO DU DOAN PRO:")
demo_test_pro(model, test_loader, classes, default_page_size=10)